# Actividad 1 – Sano vs. patológico
Columnas: `edad, peso, altura, genero, grupo`. Se entrena un clasificador, se reportan AUC/Acc/F1 y se interpreta con coeficientes, permutation importance, SHAP y PDP.

In [ ]:
!pip install -q shap
import numpy as np, pandas as pd, matplotlib.pyplot as plt, shap
from google.colab import files
up = files.upload()   # sube actividad1.csv
df = pd.read_csv(list(up.keys())[0])
print(df.shape); display(df.head()); display(df.describe()); print(df.isna().sum()); print(df.grupo.value_counts())

## 1. Exploración
Distribuciones por grupo y correlaciones (ojo: `grupo` es la etiqueta; se asume 1 = patológico, 0 = sano; verificar con el enunciado).

In [ ]:
feats = ['edad','peso','altura','genero']
fig, ax = plt.subplots(1, 3, figsize=(14,3.5))
for a, c in zip(ax, ['edad','peso','altura']):
    for g, d in df.groupby('grupo'):
        a.hist(d[c], bins=30, alpha=.5, label=f'grupo {g}')
    a.set_title(c); a.legend()
plt.show()
print(df.groupby('grupo')[feats].mean())
print(pd.crosstab(df.genero, df.grupo, normalize='index'))
display(df.corr().round(2))

## 2. Clasificadores
Se compara un modelo lineal (línea base interpretable) con modelos de árboles. Se añade `IMC = peso / (altura/100)²` como variable derivada, porque un clasificador lineal no puede construir cocientes por sí solo.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix, RocCurveDisplay

d = df.copy(); d['imc'] = d.peso / (d.altura/100)**2
def run(cols, nombre):
    X, y = d[cols], d.grupo
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, stratify=y, random_state=0)
    modelos = {'Logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
               'Random Forest': RandomForestClassifier(300, random_state=0),
               'HistGB': HistGradientBoostingClassifier(random_state=0)}
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    out, fit = [], {}
    for n, m in modelos.items():
        auc_cv = cross_val_score(m, Xtr, ytr, cv=cv, scoring='roc_auc').mean()
        m.fit(Xtr, ytr); p = m.predict_proba(Xte)[:,1]; fit[n] = m
        out.append([n, auc_cv, roc_auc_score(yte, p), accuracy_score(yte, p>.5), f1_score(yte, p>.5)])
    print(nombre); display(pd.DataFrame(out, columns=['modelo','AUC cv(train)','AUC test','Acc test','F1 test']).round(3))
    return fit, Xtr, Xte, ytr, yte

fit0, *_ = run(['edad','peso','altura','genero'], 'Variables originales')
fit1, Xtr, Xte, ytr, yte = run(['edad','peso','altura','genero','imc'], 'Con IMC')

In [ ]:
mejor = max(fit1, key=lambda n: roc_auc_score(yte, fit1[n].predict_proba(Xte)[:,1])); print('Mejor modelo:', mejor)
modelo = fit1[mejor]
RocCurveDisplay.from_estimator(modelo, Xte, yte); plt.show()
print(confusion_matrix(yte, modelo.predict(Xte)))

## 3. ¿Qué variables son más relevantes?
Tres miradas: coeficientes estandarizados (lineal), permutation importance (agnóstica, en test) y SHAP.

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
# (a) coeficientes estandarizados
lin = fit1['Logística']
print(pd.Series(lin[-1].coef_[0], index=Xtr.columns).sort_values(key=abs, ascending=False).round(3))
# (b) permutation importance del mejor modelo
pi = permutation_importance(modelo, Xte, yte, scoring='roc_auc', n_repeats=30, random_state=0)
pis = pd.Series(pi.importances_mean, index=Xte.columns).sort_values()
pis.plot.barh(xerr=pd.Series(pi.importances_std, index=Xte.columns)[pis.index], title='Permutation importance (caída de AUC)'); plt.show()
# (c) SHAP
f = lambda X: modelo.predict_proba(X)[:,1]
expl = shap.Explainer(f, Xtr)
sv = expl(Xte.iloc[:300])
shap.plots.beeswarm(sv)
shap.plots.bar(sv)

## 4. Dirección y forma del efecto (PDP)

In [ ]:
PartialDependenceDisplay.from_estimator(modelo, Xte, list(Xte.columns), kind='both', subsample=100, random_state=0)
plt.gcf().set_size_inches(14,8); plt.show()
PartialDependenceDisplay.from_estimator(modelo, Xte, [('peso','altura')]); plt.show()

## 5. Control: ¿se sostiene sin la variable dominante?
Si una sola variable concentra la importancia, se reentrena sin ella para ver cuánta información redundante hay (p. ej. IMC vs. peso+altura).

In [ ]:
top = pis.index[-1]; print('Variable más importante:', top)
_ = run([c for c in Xtr.columns if c != top], f'Sin {top}')

## 6. Conclusiones (completar con los resultados de arriba)
- **Desempeño:** AUC = __, Acc = __, F1 = __ (test). Comparar con el modelo logístico: si es similar, la relación es casi lineal.
- **Variables relevantes:** __ (coincidencia entre coeficientes, permutation y SHAP). Dirección según PDP/beeswarm: __.
- **Atribución:** discutir si la variable dominante tiene sentido fisiológico (p. ej. peso/IMC asociado a la patología) o si puede ser un sesgo/atajo (p. ej. `genero` desbalanceado entre grupos, ver crosstab de la sección 1). Un alto AUC no prueba causalidad: son datos observacionales y la importancia describe al *modelo*, no al fenómeno.
- **Limitaciones:** variables correlacionadas (peso–altura–IMC) reparten importancia y el PDP asume independencia.